# 02 — Calidad de datos

Esta es la notebook que más se mira en una entrevista técnica: no muestra código,
muestra **criterio**.

La regla que sigo: **nunca descarto sin medir el impacto, y nunca elijo entre dos
fuentes sin medir cuánto se contradicen.**

Lo que se documenta acá:

1. Qué descarta cada regla de limpieza y cuánto pesa
2. El campo `productos_ean` que no contiene el EAN
3. Unidades de medida: texto libre mezclado con un estándar internacional
4. Dos fuentes oficiales del tamaño del envase que no coinciden
5. Saltos de precio imposibles
6. Cadenas y sucursales que entran y salen del reporte

In [ ]:
import os, sys, warnings
sys.path.insert(0, "..")
# Los datos crudos viven fuera del repo (pesan GB). Si moviste la carpeta,
# cambiá esta ruta o exportá RADAR_DATA_DIR antes de abrir el notebook.
os.environ.setdefault("RADAR_DATA_DIR", os.path.expanduser("~/sepa-data"))
warnings.filterwarnings("ignore")

import duckdb, pandas as pd, matplotlib.pyplot as plt
from src import config as cfg

pd.set_option("display.max_columns", 40); pd.set_option("display.width", 150)
plt.rcParams.update({"figure.figsize": (11, 4.5), "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold"})

PARQUET = str(cfg.INTERIM / "fecha=*" / "*.parquet")
con = duckdb.connect()
dias = sorted(p.name.split("=")[1] for p in cfg.INTERIM.glob("fecha=*"))
assert dias, f"No hay datos en {cfg.INTERIM}. Corré primero: python -m src.clean"
print(f"Datos: {cfg.DATA}")
print(f"{len(dias)} días — de {dias[0]} a {dias[-1]}")

## 1. Qué descarta cada regla

`clean.py` deja registro diario de cuánto tiró cada regla. Sin esto, una regla mal
escrita puede comerse el 90% de los datos sin que nadie se entere.

In [ ]:
cal = pd.read_csv(cfg.INTERIM / "_calidad.csv")
display(cal)

desc = [c for c in cal.columns if c.startswith("desc_")]
pct = (100 * cal[desc].sum() / cal.filas_crudas.sum()).sort_values()
pct.index = [i.replace("desc_", "").replace("_", " ") for i in pct.index]
ax = pct.plot.barh(color="#95a5a6")
for i, v in enumerate(pct.values):
    ax.text(v, i, f"  {v:.3f}%", va="center", fontsize=9)
ax.set_xlabel("% de las filas crudas")
ax.set_title(f"Limpieza: se descarta el {pct.sum():.2f}% de los registros")
print(f"Filas crudas totales: {cal.filas_crudas.sum():,}")
print(f"Filas finales:        {cal.filas_finales.sum():,}")

> **Interpretación.** Un descarte muy bajo puede significar dos cosas opuestas:
> que el origen valida bien, o que mis reglas son demasiado permisivas y está
> entrando basura. Las secciones siguientes buscan basura que las reglas no ven.

## 2. El campo `productos_ean` no contiene el EAN

Uno de los hallazgos más caros del proyecto. El archivo de productos tiene una
columna llamada `productos_ean`, pero el código de barras está en `id_producto`.
`productos_ean` es la **cantidad** de EANs del producto, y vale 1 en casi todas
las filas.

```
12 | 1 | 170 | 7792180005205 | 1 | ACEITE OLIVA CAÑUELAS ... 
                ^ id_producto  ^ productos_ean (cantidad, no código)
```

Asumir lo contrario hizo que el pipeline descartara el **100%** de los registros
por "EAN inválido" y siguiera corriendo sin quejarse, escribiendo un parquet
vacío. De ahí salió el guardrail de `clean.py`: si una regla descarta más del
20% de las filas, el proceso corta y muestra ejemplos de lo que estaba tirando.

In [ ]:
# Verificación sobre un archivo crudo: productos_ean es constante, id_producto no
import glob
crudos = sorted(glob.glob(str(cfg.RAW / dias[-1] / "**" / "productos.csv"), recursive=True))
if crudos:
    raw1 = pd.read_csv(crudos[0], sep="|", dtype=str, nrows=5000, on_bad_lines="skip")
    print("Archivo:", crudos[0].split("/")[-2], "\n")
    print("productos_ean -> valores distintos:", raw1.productos_ean.nunique(),
          "| ejemplos:", raw1.productos_ean.unique()[:5].tolist())
    print("id_producto   -> valores distintos:", raw1.id_producto.nunique(),
          "| ejemplos:", raw1.id_producto.unique()[:3].tolist())
else:
    print("No hay CSV crudos a mano (se borraron o cambió la ruta). El hallazgo está documentado arriba.")

## 3. Unidades de medida: dos vocabularios en la misma columna

`productos_unidad_medida_presentacion` mezcla **texto libre** (`gr`, `GR`, `grs`,
`LTS`, `cm3`) con **códigos UN/CEFACT Rec 20**, el estándar internacional
(`kgm` = kilogram, `ltr` = litre, `cmq` = centímetro cúbico, `ea` = each).

No hay ninguna marca que distinga unos de otros. Solo `ea` eran más de 1,4
millones de filas por día que se perdían como "unidad desconocida".

`src/unidades.py` mapea los dos vocabularios a tres unidades base: kg, litro y
unidad. Lo que mide longitud o superficie (`m`, `m2`, rollos de papel y film) se
deja fuera **a propósito**: no tiene equivalencia y forzarla sería inventar.

In [ ]:
from src.unidades import unidades_no_reconocidas, _MAPA, NO_MAPEABLES

u = con.execute(f"""
    SELECT productos_unidad_medida_presentacion AS u, count(*) AS filas
    FROM read_parquet('{PARQUET}')
    WHERE u IS NOT NULL GROUP BY 1 ORDER BY filas DESC LIMIT 30
""").df()
u["mapeada"] = u.u.str.strip().str.lower().isin(_MAPA)
u["ignorada_a_proposito"] = u.u.str.strip().str.lower().isin(NO_MAPEABLES)
display(u)

sin_mapear = u[~u.mapeada & ~u.ignorada_a_proposito]
if len(sin_mapear):
    print("\nUnidades sin mapear ni ignorar — si alguna pesa, agregala a src/unidades.py:")
    display(sin_mapear)
else:
    print("\nTodas las unidades frecuentes están mapeadas o ignoradas explícitamente.")

## 4. Dos fuentes del tamaño del envase que se contradicen

**Este es el hallazgo principal de calidad de datos del proyecto.**

Para saber el tamaño de un envase hay dos caminos:

| Fuente | Ejemplo |
|---|---|
| Columnas de presentación | `cantidad = 500`, `unidad = GR` |
| La descripción del producto | `"FIDEO GUISERO 500 GR"` |

Las columnas dedicadas son poco confiables: cerca del **39%** de los envases con
peso vienen declarados como `1 un`, y hay botellas de 900 ml informadas como
`1 gr` — lo que al dividir da precios de **$97 millones por litro**.

Por eso el pipeline usa la descripción como fuente principal, deja las columnas
como respaldo, y **mide la discrepancia** en vez de elegir una en silencio.

In [ ]:
fuentes = cal[["fecha", "pct_tamano_desc", "pct_sin_tamano", "pct_fuentes_discrepan"]]
display(fuentes)

ax = fuentes.set_index("fecha").plot(marker="o")
ax.set_ylabel("% de las filas"); ax.set_ylim(0, None)
ax.set_title("Resolución del tamaño del envase, día a día")
ax.legend(["Resuelto desde la descripción", "Sin resolver",
           "Las dos fuentes se contradicen"], fontsize=9, frameon=False)

d = fuentes.pct_fuentes_discrepan
print(f"\nDiscrepancia entre fuentes: {d.min():.1f}% a {d.max():.1f}% (promedio {d.mean():.1f}%)")
print("Que sea ESTABLE entre días es lo importante: no es un archivo roto,")
print("es un problema sistemático de cómo se carga el dato en el origen.")

### Casos concretos de discrepancia

Nada convence más que ver las filas.

In [ ]:
disc = con.execute(f"""
    SELECT descripcion, cantidad_base, unidad_base,
           productos_cantidad_presentacion AS cant_declarada,
           productos_unidad_medida_presentacion AS uni_declarada,
           round(precio, 0) AS precio, round(precio_unitario, 0) AS precio_x_unidad
    FROM read_parquet('{PARQUET}')
    WHERE fuente_tamano = 'descripcion'
      AND productos_cantidad_presentacion IS NOT NULL
      AND descripcion LIKE '%GR%'
      AND try_cast(productos_cantidad_presentacion AS DOUBLE) = 1
    USING SAMPLE 12 ROWS
""").df()
display(disc)
print("Envases con peso en la descripción, declarados como cantidad = 1 en las columnas.")

## 5. Saltos de precio imposibles

Un salto de +200% de un día para el otro casi nunca es inflación: es un error de
carga o un cambio de presentación del producto con el mismo código.

In [ ]:
salt = con.execute(f"""
    WITH d AS (
        SELECT fecha, cadena, ean, any_value(descripcion) AS descripcion,
               median(precio) AS p
        FROM read_parquet('{PARQUET}') GROUP BY 1,2,3
    ), v AS (
        SELECT *, lag(p) OVER (PARTITION BY cadena, ean ORDER BY fecha) AS p_prev FROM d
    )
    SELECT fecha, cadena, substr(descripcion,1,45) AS producto,
           round(p_prev,0) AS antes, round(p,0) AS despues,
           round(100.0*(p/nullif(p_prev,0)-1),0) AS var_pct
    FROM v
    WHERE p_prev IS NOT NULL AND abs(p/nullif(p_prev,0)-1) > 1.0
    ORDER BY abs(var_pct) DESC LIMIT 20
""").df()
display(salt)

total = con.execute(f"""
    WITH d AS (SELECT fecha, cadena, ean, median(precio) AS p
               FROM read_parquet('{PARQUET}') GROUP BY 1,2,3),
         v AS (SELECT *, lag(p) OVER (PARTITION BY cadena, ean ORDER BY fecha) AS pp FROM d)
    SELECT count(*) FILTER (WHERE abs(p/nullif(pp,0)-1) > 1.0) AS saltos,
           count(*) FILTER (WHERE pp IS NOT NULL) AS pares
    FROM v
""").df().iloc[0]
print(f"\nSaltos > 100% diario: {total.saltos:,} sobre {total.pares:,} pares ({100*total.saltos/total.pares:.3f}%)")

## 6. Cadenas y sucursales intermitentes

Si una sucursal reporta 3 de 7 días, el índice se mueve por composición de la
muestra y no por precios. Esta es la limitación más importante para cualquier
comparación temporal, y hay que declararla.

In [ ]:
estab = con.execute(f"""
    WITH s AS (
        SELECT cadena, id_comercio, id_sucursal, count(DISTINCT fecha) AS dias
        FROM read_parquet('{PARQUET}') GROUP BY 1,2,3
    )
    SELECT dias, count(*) AS sucursales
    FROM s GROUP BY 1 ORDER BY dias
""").df()
display(estab)
completas = estab[estab.dias == len(dias)].sucursales.sum()
print(f"Sucursales presentes los {len(dias)} días: {completas:,} de {estab.sucursales.sum():,} "
      f"({100*completas/estab.sucursales.sum():.1f}%)")

cad = con.execute(f"""
    SELECT cadena, count(DISTINCT fecha) AS dias_reportados
    FROM read_parquet('{PARQUET}') GROUP BY 1 ORDER BY dias_reportados
""").df()
display(cad[cad.dias_reportados < len(dias)])

## Decisiones tomadas

| Regla | Motivo | Tipo |
|---|---|---|
| Precio nulo o EAN vacío | la fila no aporta información | Calidad |
| Precio fuera de `[1, 5.000.000]` | error de carga, no un precio real | Calidad |
| EAN de largo inválido | código mal formado | Calidad |
| Duplicado (comercio, sucursal, EAN) | doble reporte del mismo día | Calidad |
| Precio > 10× la mediana de ese EAN ese día | coma decimal mal cargada | Calidad |
| Unidad distinta a la esperada del item | no comparable | Comparabilidad |
| Precio unitario fuera de 0,35× – 3× la referencia | otra presentación | Comparabilidad |

**Calidad y comparabilidad se reportan por separado, y es deliberado.** Los precios
descartados por comparabilidad suelen ser correctos: una banana suelta a $799 y un
kilo a $7.050 son dos precios reales. Lo que está mal es compararlos. Mezclar los
dos conceptos inflaría el porcentaje de "datos sucios" y me dejaría mejor parado
de lo que corresponde.

## Limitaciones que se desprenden de esta notebook

> Completá con tus números:
>
> - La muestra cambia día a día (X a Y cadenas) → las comparaciones temporales
>   están condicionadas por composición.
> - Solo el Z% de las sucursales reporta los N días completos.
> - El W% de los productos tiene dos fuentes de tamaño que se contradicen.
> - El V% de las filas no tiene tamaño resoluble y queda fuera del índice.